# GSE203592 processing decisions

Task `t_40b72cca`; one biological dataset, one logical family, one physical member. This notebook is the executable reconstruction/readback layer for the append-only OBS and VAR revisions. It never rewrites X or Collections.


## Accepted facts and explicit limits

GEO and the Cell publication establish a Mus musculus direct-capture Perturb-seq assay of CD8-positive tumor-infiltrating T cells. The 48-sgRNA custom pool contains 36 guides targeting nine genes and 12 non-targeting controls. GEO, the Methods, Key Resources Table, and Table S2 do not publish nucleotide sequences, so `guide_sequence` remains `unknown`; current GPP candidates must not be projected onto the historical library. The assayed cells are primary TILs, therefore `cell_line` is not applicable even though MC38 is the tumor model.


In [ ]:
import platform
import pandas as pd
from tools.lamin_context import connect_pertdata

assert platform.system() != "Darwin", "run on the approved EU VM"
ln = connect_pertdata()
assert ln.setup.settings.instance.slug == "laminlabs/pertdata"
assert ln.setup.settings.branch.name == "jkobject"
prefix = "prism_collection/GSE203592"
task_id = "t_40b72cca"


In [ ]:
def latest(key):
    records = list(ln.Artifact.filter(key=key).all())
    records.sort(key=lambda item: (str(item.created_at), str(item.uid)))
    assert records and records[-1].is_latest
    return records[-1]

obs_artifact = latest(f"{prefix}/obs.parquet")
assert str(obs_artifact.description).startswith(f"{task_id}: source-exhaustive GSE203592 OBS")
obs = obs_artifact.load()
x = obs_artifact.features.get_values()["X"]
if isinstance(x, str):
    matches = list(ln.Artifact.filter(uid=x).all())
    x = matches[0] if len(matches) == 1 else latest(x)
assert str(x.uid) == "PhpiVnUwNAeZ26m40000"
var = x.features.get_values()["var"]
if isinstance(var, str):
    matches = list(ln.Artifact.filter(uid=var).all())
    var = matches[0] if len(matches) == 1 else latest(var)
assert str(var.description).startswith(f"{task_id}: GSE203592 mouse VAR")
var_df = var.load()


In [ ]:
assert len(obs) == 70646 and obs.index.is_unique
assert obs["obs_uuid"].is_unique and obs["original_obs_index"].is_unique
assert obs["cell_barcode"].astype("string").equals(obs["original_obs_index"].astype("string"))
assert obs["organism"].eq("Mus musculus").all()
assert obs["cell_type"].eq("CD8+ tumor infiltrating T cells").all()
assert obs["cell_line"].isna().all() and obs["cell_line_state"].eq("not_applicable").all()
assert obs["guide_sequence"].isna().all() and obs["guide_sequence_state"].eq("unknown").all()
assert (obs["is_control"].fillna(False).astype(bool).to_numpy() == obs["condition"].astype("string").eq("control").fillna(False).to_numpy(dtype=bool)).all()
assert obs["n_counts"].notna().all() and obs["n_genes"].notna().all() and obs["pct_mito"].notna().all()
assert obs["timepoint"].eq(21600.0).all()


In [ ]:
assert len(var_df) == 31053 and var_df.index.is_unique
stable = var_df["stable_feature_id"].astype("string")
mapped = stable.dropna()
assert mapped.str.fullmatch(r"ENSMUSG\d{11}").all()
assert mapped.is_unique and stable.notna().mean() >= 0.99
assert var_df["organism"].eq("Mus musculus").all()
assert var_df["feature_index"].notna().all() and var_df["feature_index"].is_unique
assert not stable.str.fullmatch(r"ENSG\d{11}", na=False).any()


In [ ]:
for key, expected_count in [("pert-gym/additions/20260621", 996), ("pert-gym/canonical/20260621", 1056)]:
    collection = list(ln.Collection.filter(key=key).all())
    assert len(collection) == 1
    members = list(collection[0].artifacts.only("uid", "key").all())
    assert len(members) == expected_count
    assert sum(str(item.key) == f"{prefix}/obs.parquet" for item in members) == 1

pd.DataFrame({"obs_uid": [str(obs_artifact.uid)], "x_uid": [str(x.uid)], "var_uid": [str(var.uid)], "mapped_mouse_ids": [int(stable.notna().sum())]})
